# Fase 2b: Variante CT con Tversky ponderada

Notebook dedicado exclusivamente a probar una variante ligera de segmentacion CT. No ejecuta CXR ni los baselines ya guardados.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import config
from src.data.segmentation import build_ct_segmentation_dataframe, split_segmentation_dataframe
from src.training.segmentation_experiment import (
    existing_segmentation_artifact,
    get_device,
    make_segmentation_run_config,
    save_segmentation_artifacts,
    seed_everything,
    summarize_segmentation_results,
    train_and_evaluate_segmentation,
)

pd.set_option('display.max_columns', 50)
seed_everything(config.RANDOM_SEED)
device = get_device()
device

## 2. Configuracion de la variante

Esta configuracion guarda un experimento nuevo y no pisa `ct_attention_unet_segmentation_full`.

In [ ]:
RUN_MODE = 'full'
RESUME_EXISTING = True

run_config = make_segmentation_run_config(
    dataset_name='ct',
    architecture='attention_unet',
    run_mode=RUN_MODE,
    image_size=config.CT_IMAGE_SIZE,
    in_channels=1,
    variant_name='tversky_pos30_bf16_thr',
    loss_name='weighted_tversky_bce',
    bce_weight=0.2,
    dice_weight=0.8,
    pos_weight=30.0,
    tversky_alpha=0.3,
    tversky_beta=0.7,
    optimize_threshold=True,
    base_features=16,
    batch_size=8,
    epochs=12,
    early_stopping_patience=4,
    learning_rate=1e-4,
)

seg_models_dir = config.MODELS_DIR / 'segmentation' / 'ct'
seg_results_dir = PROJECT_ROOT / 'results' / 'segmentation' / 'ct'
processed_seg_dir = config.CT_DIR / 'processed_segmentation_slices'
seg_models_dir.mkdir(parents=True, exist_ok=True)
seg_results_dir.mkdir(parents=True, exist_ok=True)

print(f'Experimento: {run_config.experiment_name}_{run_config.run_mode}')
print(f'Loss: {run_config.loss_name}')
print(f'pos_weight: {run_config.pos_weight}')
print(f'base_features: {run_config.base_features}')
print(f'epochs: {run_config.epochs}')
print(f'batch_size: {run_config.batch_size}')

## 3. Datos CT y split por estudio

In [ ]:
ct_seg_df = build_ct_segmentation_dataframe(
    ct_dir=config.CT_DIR,
    output_dir=processed_seg_dir,
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)

train_df, val_df, test_df = split_segmentation_dataframe(
    ct_seg_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)

split_summary = pd.DataFrame([
    {'split': 'train', 'slices': len(train_df), 'studies': train_df['study_id'].nunique()},
    {'split': 'val', 'slices': len(val_df), 'studies': val_df['study_id'].nunique()},
    {'split': 'test', 'slices': len(test_df), 'studies': test_df['study_id'].nunique()},
])
display(split_summary)
print(f'Total CT slices con mascara positiva: {len(ct_seg_df)}')

## 4. Entrenamiento

In [ ]:
if RESUME_EXISTING and existing_segmentation_artifact(run_config, seg_models_dir, seg_results_dir):
    print('Saltado: esta variante ya tiene modelo y summary guardados.')
    latest_result = None
else:
    latest_result = train_and_evaluate_segmentation(
        run_config=run_config,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        device=device,
    )
    saved_paths = save_segmentation_artifacts(run_config, latest_result, seg_models_dir, seg_results_dir)
    print(saved_paths)

## 5. Comparacion con baselines CT

In [ ]:
summary_paths = sorted(seg_results_dir.glob('*_summary.json'))
summary_df = summarize_segmentation_results(summary_paths)
cols = ['experiment', 'variant_name', 'loss_name', 'dice', 'iou', 'pixel_accuracy', 'threshold', 'hyperparameters']
display(summary_df.sort_values('dice', ascending=False)[[c for c in cols if c in summary_df.columns]])

## 6. Visualizacion cualitativa

In [ ]:
if latest_result is not None:
    examples = latest_result['examples'][:4]
    threshold = latest_result.get('threshold', 0.5)
    fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
    if len(examples) == 1:
        axes = axes[None, :]
    for row_idx, example in enumerate(examples):
        image = example['image'].squeeze().numpy()
        mask = example['mask'].squeeze().numpy()
        pred = (example['prediction'].squeeze().numpy() >= threshold).astype(float)
        axes[row_idx, 0].imshow(image, cmap='gray')
        axes[row_idx, 0].set_title('Imagen')
        axes[row_idx, 1].imshow(mask, cmap='gray')
        axes[row_idx, 1].set_title('Mascara real')
        axes[row_idx, 2].imshow(pred, cmap='gray')
        axes[row_idx, 2].set_title(f'Prediccion t={threshold:.2f}')
        for ax in axes[row_idx]:
            ax.axis('off')
    plt.tight_layout()
else:
    print('No hay entrenamiento nuevo en esta ejecucion.')